In [4]:
import sys
sys.path.insert(0, '..')
from panel_exp.panel_data import long_df_to_paneldataset, PanelDataset, TimePeriod
from panel_exp.panel_data import long_df_to_paneldataset, PanelDataset, TimePeriod
from panel_exp.design import CompleteRandomization, ThinningDesign, Rerandomization, QuickBlock
from panel_exp.design.design_metrics import imbalance

import numpy as np
import pandas as pd

In [ ]:
def sum_max_k(v, k):
    if len(v) >= k:
        return np.sum(np.sort(v)[-k:]), np.argsort(v)[-k:]
    else:
        return np.sum(np.sort(v)[-len(v):]), np.argsort(v)[-len(v):]


# the matching design is a function that takes as input a panel_data object
# and returns a treatment assignment for the given time period
def matching_design(panel_data, k=1):

    # access the wide data frame from panel_data
    df = panel_data.wide_df
    nunits = df.shape[0]
    ntime = df.shape[1]

    covariate_matrix = panel_data.wide_data.values

    # obtain a similarity matrix ffrom covariate_matrix by taking pairwise l2 distances between rows of covariate_matrix
    similarity_matrix = np.zeros((nunits, nunits))
    for i in range(nunits):
        for j in range(nunits):
            similarity_matrix[i, j] = np.linalg.norm(covariate_matrix[i, :] - covariate_matrix[j, :])
    
    # obtain a greedy matching by taking the argmax of the similarity matrix and then removing the matched rows and columns from the similarity matrix
    # repeat until all units are matched
    
    matched_vertices = set([i for i in range(nunits)])
    matching = {}
    max_similarity = 0.0
    while len(matched_vertices) > 0:
        for i in matched_vertices:
            similarity_vi, vi_matched_vertices = sum_max_k(similarity_matrix[i, :], k)
            if similarity_vi > max_similarity:
                max_similarity = similarity_vi
                max_similarity_vertex = i
                max_similarity_matched_vertices = vi_matched_vertices
            
        matching[max_similarity_vertex] = max_similarity_matched_vertices
        matched_vertices.remove(max_similarity_vertex)
        for i in max_similarity_matched_vertices:
            matched_vertices.remove(i)

    return matching

